# ULTRON Day 1 — Foundation + Stock Selection Engine

**What this notebook does:** sets up ULTRON in Google Colab, shows the data adapter, validates data, calculates features,
ranks the NIFTY 50 + BANK NIFTY stocks, and saves the shortlist.

> ⚠️ **IMPORTANT — SYNTHETIC DEMO DATA.** No market-data provider has been selected yet, and ULTRON never invents live data.
> This notebook therefore uses **clearly labelled random (synthetic) data** so you can see every step working.
> Demo symbols are named `DEMO-…` on purpose. **The shortlist you see here is NOT a real stock pick.**
>
> ULTRON is for research and paper trading only. It never places orders. A high rank does not guarantee profit.

**How to use:** run the cells from top to bottom (Runtime → Run all, or press Shift+Enter on each cell). Each cell has a short explanation above it.

## Cell 1 — Environment setup
This cell gets the ULTRON code into Colab. Choose ONE mode in `SETUP_MODE`:

* `"github"` (default) — downloads your repository `prakhar261106/ULTRON-INDIA` (use this **after** you uploaded the Day 1 files to GitHub).
* `"zip"` — you upload `ULTRON_Day1_deliverables.zip` from your computer (use this if the files are not on GitHub yet).
* `"local"` — the code is already in the current folder (advanced).

In [ ]:
import os, sys, pathlib, subprocess, zipfile

SETUP_MODE = "github"      # "github" | "zip" | "local"
REPO_URL   = "https://github.com/prakhar261106/ULTRON-INDIA.git"
REPO_DIR   = pathlib.Path("/content/ULTRON-INDIA")
MARKER     = "shared/contracts/engine_output.py"   # a Day 1 file used to check the setup worked

if SETUP_MODE == "github":
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
elif SETUP_MODE == "zip":
    from google.colab import files
    print("Choose ULTRON_Day1_deliverables.zip in the file picker...")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(REPO_DIR)
elif SETUP_MODE == "local":
    REPO_DIR = pathlib.Path.cwd()
else:
    raise ValueError("SETUP_MODE must be 'github', 'zip' or 'local'")

if not (REPO_DIR / MARKER).exists():
    raise FileNotFoundError(
        "The Day 1 files were not found. If you used 'github', upload the Day 1 files to GitHub first "
        "(see documentation/DAY1_UPLOAD_GUIDE.md) or set SETUP_MODE = 'zip'.")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Working folder:", os.getcwd())
print("OK - Day 1 files found.")

## Cell 2 — Install dependencies
Installs the few libraries ULTRON needs (listed in `requirements.txt`). Colab already has most of them, so this is quick.
If Colab shows a "restart session" message, restart and run again from Cell 1.

In [ ]:
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                        capture_output=True, text=True)
print(result.stdout[-1500:] or "Dependencies are ready.")
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("pip install failed - read the message above.")

## Cell 3 — Imports and version check
Loads the ULTRON modules and prints library versions (useful if something ever breaks).

In [ ]:
import json
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

from config.settings import load_config
from config.universe import load_universe, Universe, UniverseMember
from engines.stock_selection import StockSelectionEngine
from engines.stock_selection.features import compute_symbol_features
from shared.contracts import (IST, Timeframe, EngineOutput, UnconfiguredAdapter, ProviderNotConfiguredError)
from shared.data_validation import validate_candles, complete_bars
from shared.logging import get_logger
from tests.synthetic_data import SyntheticMarketAdapter, weekdays_ending   # SYNTHETIC data tools (tests only)

print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)
log = get_logger("ultron.notebook")

## Cell 4 — Project configuration
All settings live in **`config/ultron_config.yaml`** (market hours, data-quality limits, liquidity filters, ranking weights).
Nothing is hidden in the code. Note `data_provider.name` is empty: **no provider is selected yet.**

In [ ]:
cfg = load_config()
print("Session:", cfg.market.session_open, "-", cfg.market.session_close, "|", cfg.market.timezone)
print("Timeframes:", [t.value for t in cfg.timeframes])
print("Data provider:", cfg.data_provider.name, "(None = not selected yet)")
print("Shortlist size (top_n):", cfg.stock_selection.top_n)
print("Min avg daily traded value (INR):", f"{cfg.stock_selection.min_avg_daily_traded_value_inr:,.0f}")
print("Ranking weights:")
print(pd.Series(cfg.stock_selection.normalised_weights()).round(3).to_string())

## Cell 5 — Stock universe (NIFTY 50 + BANK NIFTY)
Loads the universe, removes duplicates (e.g. HDFC Bank is in both indices) and respects membership dates.
**Always verify the list** against official NSE Indices files — `config/universe.yaml` explains what was checked.

In [ ]:
TODAY = datetime.now(IST).date()
END_DATE = weekdays_ending(TODAY, 1)[0]                      # latest weekday (no holiday logic)
AS_OF = datetime(END_DATE.year, END_DATE.month, END_DATE.day, 15, 30, tzinfo=IST)

universe = load_universe(END_DATE)
raw_total = len(load_universe(END_DATE, indices=("NIFTY50",)).symbols) + len(load_universe(END_DATE, indices=("BANKNIFTY",)).symbols)
print(f"As of {END_DATE}: {len(universe.symbols)} unique symbols ({raw_total - len(universe.symbols)} duplicates removed)")
for w in universe.warnings: print("WARNING:", w)
pd.DataFrame([{"symbol": m.symbol, "sector": m.sector, "indices": "+".join(m.indices)} for m in universe.members]).head(12)

## Cell 6 — Market data adapter demonstration
ULTRON talks to data through one **provider-independent interface** (`MarketDataAdapter`). Two things to see:

1. With **no provider configured**, ULTRON *refuses* to return data (it never silently substitutes fake data).
2. A clearly labelled **SYNTHETIC** adapter, used here only so the notebook can run. Real providers (Dhan, Zerodha, ...) will be added later as small adapter classes.

*Have your own exported candle files?* Put `SYMBOL_15m.csv` files (columns `timestamp,open,high,low,close,volume`, with `+05:30` in timestamps) into `data/raw/` and use `CsvMarketDataAdapter("data/raw")` instead.

In [ ]:
# 1) No provider configured -> a clear error, never fake data
try:
    UnconfiguredAdapter().get_historical_candles("RELIANCE", "15m", AS_OF - timedelta(days=5), AS_OF)
except ProviderNotConfiguredError as e:
    print("Expected refusal ->", str(e)[:140], "...")

# 2) SYNTHETIC demo data. Symbols are renamed DEMO-<SYMBOL> so nobody mistakes them for real picks.
TIMEFRAME = "15m"          # try "5m" or "1h" too
demo_members = [UniverseMember(symbol="DEMO-" + m.symbol, sector=m.sector, indices=m.indices) for m in universe.members]
demo_universe = Universe(as_of=END_DATE, members=demo_members, membership_history_verified=False,
                         warnings=universe.warnings)
adapter = SyntheticMarketAdapter(demo_universe.symbols, END_DATE, n_sessions=25, seed=42,
                                 sector_of=demo_universe.sector_of)
candles = adapter.get_historical_candles("DEMO-RELIANCE", TIMEFRAME, AS_OF - timedelta(days=40), AS_OF)
print(f"\nSYNTHETIC candles for DEMO-RELIANCE ({TIMEFRAME}): {candles.shape[0]} rows")
candles.head()

## Cell 7 — Data validation
Before any number is used, the candles are checked: missing values, duplicates, invalid OHLC, missing candles, stale data, wrong timezone...
First on good data, then on a deliberately damaged copy so you can see the problems being caught.

In [ ]:
good = validate_candles(candles, TIMEFRAME, AS_OF, cfg.market, cfg.data_quality, symbol="DEMO-RELIANCE")
print("GOOD data  -> valid:", good.is_valid, "| status:", good.status.value, "| quality score:", good.quality_score)

bad = candles.copy()
bad.loc[5, "close"] = np.nan                                  # a missing value
bad.loc[8, "high"] = bad.loc[8, "low"] - 1                    # impossible OHLC
bad = pd.concat([bad, bad.iloc[[20]]]).sort_values("timestamp", kind="stable").reset_index(drop=True)  # duplicate candle
rep = validate_candles(bad, TIMEFRAME, AS_OF, cfg.market, cfg.data_quality, symbol="DEMO-RELIANCE")
print("DAMAGED    -> valid:", rep.is_valid, "| status:", rep.status.value)
pd.DataFrame([i.model_dump() for i in rep.issues])[["severity", "code", "count", "message"]]

## Cell 8 — Feature calculation (one stock)
Shows the numbers the ranking is built from, for a single stock. Only candles whose bar has **closed** at the "as of" time are used (no look-ahead).

In [ ]:
index_df = adapter.get_historical_candles("NIFTY 50", TIMEFRAME, AS_OF - timedelta(days=40), AS_OF)
closed = complete_bars(candles, TIMEFRAME, AS_OF, cfg.market)
feat = compute_symbol_features(closed, index_df, TIMEFRAME, cfg.stock_selection, cfg.market, "DEMO-RELIANCE")
print("Latest session:", feat.current_session, "| prior sessions used:", feat.n_prior_sessions)
pd.Series(feat.values).rename("value").to_frame()

## Cell 9 — Stock ranking
Runs the full engine on all 58 demo symbols. To show the safety filters working, two symbols are damaged on purpose:
`DEMO-ITC` gets an illiquid (tiny volume) history and `DEMO-TCS` gets corrupted candles. Both should appear in the **excluded** list.

In [ ]:
engine = StockSelectionEngine(cfg, logger=log)

stocks = {s: adapter.get_historical_candles(s, TIMEFRAME, AS_OF - timedelta(days=40), AS_OF) for s in demo_universe.symbols}
stocks["DEMO-ITC"]["volume"] = (stocks["DEMO-ITC"]["volume"] / 1000).round()      # make it illiquid
stocks["DEMO-TCS"].loc[10, "high"] = stocks["DEMO-TCS"].loc[10, "low"] - 5         # corrupt it

result = engine.run(stocks, index_df, TIMEFRAME, AS_OF, demo_universe, data_label="SYNTHETIC_DEMO_DATA")
print(f"Universe: {result.universe_size} | eligible: {result.n_eligible} | excluded: {len(result.exclusions)} | shortlisted: {len(result.shortlist)}")
result.ranking[["rank", "symbol", "sector", "composite_score", "feature_coverage"]].head(15)

## Cell 10 — Shortlist output
The top 10. Every row is a validated **EngineOutput** — the common format all ULTRON engines will use.
`direction` is always NEUTRAL here: this engine says *"look at these stocks"*, not *"buy"* or *"sell"*.

In [ ]:
shortlist_table = pd.DataFrame([{
    "rank": o.metadata["rank"], "symbol": o.symbol, "score": round(o.score, 3), "confidence": round(o.confidence, 3),
    "direction": o.direction.value, "risk_flags": ", ".join(o.risk_flags)} for o in result.shortlist])
display(shortlist_table)

print("\nFull contract output for the #1 stock (first 12 evidence items shown):")
one = result.shortlist[0].to_json_dict()
one["supporting_evidence"] = one["supporting_evidence"][:12]
print(json.dumps(one, indent=2)[:3500])

## Cell 11 — Excluded symbols, warnings and methodology
ULTRON always tells you **why** a stock was left out, and which limits apply.

In [ ]:
display(pd.DataFrame([{"symbol": e.symbol, "reasons": " | ".join(e.reasons)} for e in result.exclusions]))
print("\nWarnings:")
for w in result.warnings: print(" -", w)
print("\n" + result.methodology)

## Cell 12 — Run the unit tests
Runs the full automated test suite (data validation, ranking, duplicate symbols, liquidity filter, top-10 selection, output contract, ...).
**Expected result:** `86 passed` and no failures (if you used the ZIP mode there is no git history, so you will see `85 passed, 1 skipped` — that is normal).

In [ ]:
t = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(t.stdout[-3000:])
if t.returncode != 0:
    print(t.stderr[-2000:])
assert t.returncode == 0, "Some tests FAILED - copy the output above into the chat."
print("ALL TESTS PASSED")

## Cell 13 — Save the results
Writes four files to `data/processed/stock_selection/` (this folder is git-ignored, so results stay in Colab and never reach GitHub by accident):
the full ranking (CSV), the shortlist (JSON), the exclusions (JSON) and a run summary (JSON).

Colab storage is **temporary**. To keep the files, run the optional Google Drive cell below or download them.

In [ ]:
out_dir = cfg.path("processed_data") / "stock_selection"
paths = result.save(out_dir, prefix=f"SYNTHETIC_DEMO_{TIMEFRAME}")
for name, p in paths.items():
    print(f"{name:11s} -> {p.relative_to(REPO_DIR)}  ({p.stat().st_size:,} bytes)")

In [ ]:
# OPTIONAL: copy the results to your Google Drive so they survive after Colab closes.
# Remove the leading '#' on the lines below, run the cell and allow access when asked.
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil; shutil.copytree(out_dir, "/content/drive/MyDrive/ULTRON/stock_selection", dirs_exist_ok=True)

## What you have now

* A validated **shared output contract**, **configuration**, **logging** with secret redaction, and **data validation**.
* A provider-independent **data adapter** interface (no provider chosen yet, no fake live data).
* An independent **Stock Selection Engine** with tests.

**Not done / not verified:** the engine has **not** been run on real NSE data (no provider yet), and nothing here says anything about profitability.

**Next step:** tell Claude which data provider you will use (or that you will export CSV files), and whether the Day 1 tests passed in your Colab.